Entire code by Arshnoor

In [ ]:
!pip install requests beautifulsoup4 pymongo lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 14.1 MB/s eta 0:00:00


In [ ]:
!pip install python-dotenv

In [ ]:
#import libraries
import time
import random
import re
import json
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

In [ ]:
#base URL for The Infatuation website.
BASE = "https://www.theinfatuation.com"
INDEX_URL = f"{BASE}/san-francisco/neighborhoods/san-francisco"

#Standard HTTP headers
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

#one session for all requests
session = requests.Session()
session.headers.update(HEADERS)

#request with a small random delay
def polite_get(url, min_sleep=1.0, max_sleep=2.0):
    time.sleep(random.uniform(min_sleep, max_sleep))
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    return resp

#clean text
def normalize_text(x):
    if not x:
        return None
    x = re.sub(r"\s+", " ", x).strip()
    return x or None

#check if rating
def is_rating(text):
    if not text:
        return False
    text = text.strip()
    return bool(re.fullmatch(r"\d(?:\.\d)?", text))

#get neighborhood guide urls
def get_neighborhood_guide_urls():
    html = polite_get(INDEX_URL).text
    soup = BeautifulSoup(html, "html.parser")

    urls = set()

    # Find all <a> tags
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if "/guides/" not in href:
            continue

        full = urljoin(BASE, href)

        if "/san-francisco/" not in full:
            continue

        urls.add(full)

    #convert unique url to a sorted list
    return sorted(urls)

In [ ]:
#get guide URL
guide_urls = get_neighborhood_guide_urls()
len(guide_urls), guide_urls[:10]

(13,
 ['https://www.theinfatuation.com/san-francisco/guides/best-alamo-square-restaurants-sf',
  'https://www.theinfatuation.com/san-francisco/guides/best-mission-restaurants-sf',
  'https://www.theinfatuation.com/san-francisco/guides/best-restaurants-in-japantown-sf',
  'https://www.theinfatuation.com/san-francisco/guides/best-restaurants-in-the-sunset',
  'https://www.theinfatuation.com/san-francisco/guides/best-restaurants-near-fishermans-wharf-and-ghirardelli-square',
  'https://www.theinfatuation.com/san-francisco/guides/best-richmond-restaurants-sf',
  'https://www.theinfatuation.com/san-francisco/guides/embarcadero-ferry-building-restaurants-sf',
  'https://www.theinfatuation.com/san-francisco/guides/fidi-lunch-restaurants-sf',
  'https://www.theinfatuation.com/san-francisco/guides/noe-valley-restaurants',
  'https://www.theinfatuation.com/san-francisco/guides/north-beach-restaurants-san-francisco'])

In [ ]:
#get restaurant details
def extract_restaurant_cards_from_guide(url):
    html = polite_get(url).text
    soup = BeautifulSoup(html, "html.parser")

    restaurants = []
    seen = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()

        #keep only review links
        if "/reviews/" not in href:
            continue


        name = normalize_text(a.get_text(" ", strip=True))
        if not name:
            continue

        review_url = urljoin(BASE, href)

        #find the full restaurant card
        container = a
        for _ in range(6):
            container = container.parent
            if container is None:
                break

            text_blob = normalize_text(container.get_text(" ", strip=True)) or ""
            has_maps = bool(container.find("a", href=lambda x: x and "google.com/maps" in x))
            has_perfect_for = bool(container.find("a", href=lambda x: x and "/perfect-for/" in x))
            has_cuisine = bool(container.find("a", href=lambda x: x and "/cuisines/" in x))
            has_neighborhood = bool(container.find("a", href=lambda x: x and "/neighborhoods/" in x))

            #Use only if it is a full restaurant card
            if len(text_blob) > 80 and (has_maps or has_perfect_for or has_cuisine or has_neighborhood):
                break

        if container is None:
            continue

        #avoid duplicate
        key = (name, review_url)
        if key in seen:
            continue
        seen.add(key)

        #address
        address = None
        maps_link = container.find("a", href=lambda x: x and "google.com/maps" in x)
        if maps_link:
            address = normalize_text(maps_link.get_text(" ", strip=True))

        #cuisines
        cuisines = []
        for c in container.find_all("a", href=lambda x: x and "/cuisines/" in x):
            txt = normalize_text(c.get_text(" ", strip=True))
            if txt and txt not in cuisines:
                cuisines.append(txt)

        #neighborhoods
        neighborhoods = []
        for n in container.find_all("a", href=lambda x: x and "/neighborhoods/" in x):
            txt = normalize_text(n.get_text(" ", strip=True))
            if txt and txt not in neighborhoods:
                neighborhoods.append(txt)

         #perfect for
        perfect_for = []
        for p in container.find_all("a", href=lambda x: x and "/perfect-for/" in x):
            txt = normalize_text(p.get_text(" ", strip=True))
            if txt and txt not in perfect_for:
                perfect_for.append(txt)

        #rating
        rating = None
        for s in container.stripped_strings:
            s = s.strip()
            if is_rating(s):
                rating = float(s)
                break

        #description
        texts = [normalize_text(t) for t in container.stripped_strings]
        texts = [t for t in texts if t] # Filter out empty strings.

        excluded = {name}
        if address:
            excluded.add(address)
        excluded.update(cuisines)
        excluded.update(neighborhoods)
        excluded.update(perfect_for)
        if rating is not None:
            excluded.add(str(rating).rstrip("0").rstrip("."))

        #get description
        description = None
        for t in texts:
            if t in excluded:
                continue
            if len(t) >= 30:
                description = t
                break

        #append all
        restaurants.append({
            "name": name,
            "review_url": review_url,
            "guide_url": url,
            "address": address,
            "cuisines": cuisines,
            "neighborhoods": neighborhoods,
            "perfect_for": perfect_for,
            "rating": rating,
            "description": description,
        })
    #return list of restaurants
    return restaurants

In [ ]:
#test one guide
test_url = guide_urls[0]
sample = extract_restaurant_cards_from_guide(test_url)
len(sample)

16

In [ ]:
#sample records
sample[:3]

[{'name': 'Lucinda’s Deli & More',
  'review_url': 'https://www.theinfatuation.com/san-francisco/reviews/lucindas-deli-and-more',
  'guide_url': 'https://www.theinfatuation.com/san-francisco/guides/best-alamo-square-restaurants-sf',
  'address': '535 Scott St, San Francisco, CA 94117',
  'cuisines': ['Sandwiches'],
  'neighborhoods': ['Nopa'],
  'perfect_for': ['Lunch', 'Serious Takeout Operation'],
  'rating': None,
  'description': None},
 {'name': 'READ THE REVIEW',
  'review_url': 'https://www.theinfatuation.com/san-francisco/reviews/lucindas-deli-and-more',
  'guide_url': 'https://www.theinfatuation.com/san-francisco/guides/best-alamo-square-restaurants-sf',
  'address': '535 Scott St, San Francisco, CA 94117',
  'cuisines': ['Sandwiches'],
  'neighborhoods': ['Nopa'],
  'perfect_for': ['Lunch', 'Serious Takeout Operation'],
  'rating': 8.2,
  'description': 'royalty. And you only have to look as far as Alamo Square for evidence—practically every other person is gazing lovingly at

In [ ]:
#Collect restaurant data
all_docs = []

#loop through each guide URL
for guide_url in guide_urls:
    try:
        docs = extract_restaurant_cards_from_guide(guide_url)
        print(f"{guide_url} -> {len(docs)} restaurants")
        all_docs.extend(docs)
    except Exception as e:
        print(f"FAILED: {guide_url} -> {e}")

https://www.theinfatuation.com/san-francisco/guides/best-alamo-square-restaurants-sf -> 16 restaurants
https://www.theinfatuation.com/san-francisco/guides/best-mission-restaurants-sf -> 54 restaurants
https://www.theinfatuation.com/san-francisco/guides/best-restaurants-in-japantown-sf -> 32 restaurants
https://www.theinfatuation.com/san-francisco/guides/best-restaurants-in-the-sunset -> 49 restaurants
https://www.theinfatuation.com/san-francisco/guides/best-restaurants-near-fishermans-wharf-and-ghirardelli-square -> 24 restaurants
https://www.theinfatuation.com/san-francisco/guides/best-richmond-restaurants-sf -> 53 restaurants
https://www.theinfatuation.com/san-francisco/guides/embarcadero-ferry-building-restaurants-sf -> 47 restaurants
https://www.theinfatuation.com/san-francisco/guides/fidi-lunch-restaurants-sf -> 49 restaurants
https://www.theinfatuation.com/san-francisco/guides/noe-valley-restaurants -> 32 restaurants
https://www.theinfatuation.com/san-francisco/guides/north-beach

In [ ]:
#remove duplicates
def normalize_name(name):
    if not name:
        return None
    name = name.lower()
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\b(?:inc|llc|ltd|co|corp|corporation|company)\b", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name or None

#keep only the first record
deduped_docs = []
seen_review_urls = set()

for doc in all_docs:
    review_url = doc.get("review_url")
    if not review_url or review_url in seen_review_urls:
        continue
    seen_review_urls.add(review_url)
    deduped_docs.append(doc)

#add normalized_name
for doc in deduped_docs:
    doc["normalized_name"] = normalize_name(doc.get("name"))

#save to json
with open("infatuation_restaurants.json", "w", encoding="utf-8") as f:
    json.dump(deduped_docs, f, ensure_ascii=False, indent=2)

#summary
total_scraped = len(all_docs)
after_dedup = len(deduped_docs)
with_ratings = sum(1 for doc in deduped_docs if doc.get("rating") is not None)
with_addresses = sum(1 for doc in deduped_docs if doc.get("address"))
unique_neighborhoods = sorted({
    neighborhood
    for doc in deduped_docs
    for neighborhood in (doc.get("neighborhoods") or [])
    if neighborhood
})

print(f"Total restaurants scraped across all guides: {total_scraped}")
print(f"Restaurants after deduplication: {after_dedup}")
print(f"Restaurants with ratings: {with_ratings}")
print(f"Restaurants with addresses: {with_addresses}")
print(f"Unique neighborhoods represented: {len(unique_neighborhoods)}")
print("Saved file: infatuation_restaurants.json")


Total restaurants scraped across all guides: 511
Restaurants after deduplication: 247
Restaurants with ratings: 22
Restaurants with addresses: 247
Unique neighborhoods represented: 16
Saved file: infatuation_restaurants.json
